In [1]:
# All external paths are defined here.
# The notebook is intended to be run from a directory next to ../dca/.

FORMAT_NOTEBOOK = "../dca/input/Format.ipynb"
DATA_DIRECTORY = "../dca/input/data/"
SIM_DIRECTORY = "../dca/../sim/output/fall25/dca/"

DATA_FILE = DATA_DIRECTORY + "m_ee_DCA_900.root"  # Au+Au 0-93%

SIM_FILES = {
    "pi0":          SIM_DIRECTORY + "pi0_gee_100M_v01.root",
    "phi":          SIM_DIRECTORY + "phi_25M_v04.root",
    "phi_etaee":    SIM_DIRECTORY + "phi_etaee_v00.root",
    "jpsi":         SIM_DIRECTORY + "jpsi_25M_v06.root",
    "ccbar":        SIM_DIRECTORY + "ccbar_soft_v30.root",
    "bbbar":        SIM_DIRECTORY + "bbbar_v03.root",
    "omega_ee":     SIM_DIRECTORY + "omega_ee_v01.root",
    "omega_pi0ee":  SIM_DIRECTORY + "omega_pi0ee_v00.root",
    "thermal":      SIM_DIRECTORY + "thermal_300_v04.root",
    "eta":          SIM_DIRECTORY + "eta_gee_v00.root",
    "rho":          SIM_DIRECTORY + "rho_ee_v00.root",
    "etap":         SIM_DIRECTORY + "etap_gee_v00.root",
    "psip":         SIM_DIRECTORY + "psip_ee_v00.root",
}


In [2]:
#%%capture
%run $FORMAT_NOTEBOOK

import math
import numpy as np
import pandas as pd
import ROOT as root
from scipy.optimize import least_squares

root.TH1.AddDirectory(False)
root.gErrorIgnoreLevel = root.kFatal
%jsroot on


/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

# Direct virtual photons in 0–93% Au+Au at \(\sqrt{s_{NN}}=200\) GeV

The hadronic cocktail may contain any selected light-hadron components.
Thermal radiation and heavy flavor are explicitly excluded from this fit.


In [3]:
# ---------------- Analysis configuration ----------------

CENTRALITY_LABEL = "0-93% Au+Au"
NCOLL_AUAU_MB = 251.0  # from m_ee_decomposer74

SUBTRACTION_SCALE = 0.955
DATA_OPTION = 14


PT_BINS = [
    (0.8, 1.2),
    (1.2, 1.6),
    (1.6, 2.0),
    (2.0, 2.8),
    (2.8, 3.6),
    (3.6, 4.2),
    (4.2, 6.0)
]


MASS_PLOT_RANGE = (0.0, 0.5)
MASS_FIT_RANGE = (0.03, 0.40)
LOWMASS_RANGE = (0.0, 0.030)

# The mass fit returns a dilepton fraction in MASS_FIT_RANGE.
# Generated templates are used to extrapolate it to this wider mass range.
GENERATED_EXTRAPOLATION_RANGE = (2.0 * 0.00051099895, 0.50)

# Interpretation of the extrapolated ratio:
# "full_pair_ratio" gives the direct/cocktail converted-pair ratio in the
# generated extrapolation range. It is photon-equivalent only if the generated
# templates have consistent normalization per source photon.
GENERATOR_RATIO_MODE = "full_pair_ratio"

COCKTAIL_NORMALIZATION_MODE = "original"
# alternatives:
# "fixed_lowmass"
# "seed_from_lowmass"

# Select the light-hadron cocktail here.
COCKTAIL_COMPONENTS = ["pi0", "eta"]
# examples:
# ["pi0", "eta", "omega_ee", "omega_pi0ee", "rho", "etap", "phi_etaee"]
# ["pi0", "eta", "omega_ee", "omega_pi0ee", "rho", "etap", "phi"]

DIRECT_TEMPLATE_SOURCE = "eta"

# Explicitly prohibited in the low-mass direct-photon cocktail.
FORBIDDEN_COCKTAIL_COMPONENTS = {"thermal", "ccbar", "bbbar"}

bad_components = FORBIDDEN_COCKTAIL_COMPONENTS.intersection(COCKTAIL_COMPONENTS)
if bad_components:
    raise ValueError(
        "Do not include thermal or heavy flavor in this fit: "
        f"{sorted(bad_components)}"
    )


In [4]:
# Component mapping and normalizations copied from m_ee_decomposer74.

COMPONENTS = {
    "pi0":         {"isim": 0,  "label": "#pi^{0} Dalitz"},
    "phi":         {"isim": 1,  "label": "#phi"},
    "phi_etaee":   {"isim": 2,  "label": "#phi #rightarrow #eta e^{+}e^{-}"},
    "jpsi":        {"isim": 3,  "label": "J/#psi"},
    "ccbar":       {"isim": 4,  "label": "c#bar{c}"},
    "bbbar":       {"isim": 5,  "label": "b#bar{b}"},
    "omega_ee":    {"isim": 6,  "label": "#omega #rightarrow e^{+}e^{-}"},
    "omega_pi0ee": {"isim": 7,  "label": "#omega #rightarrow #pi^{0}e^{+}e^{-}"},
    "thermal":     {"isim": 8,  "label": "thermal"},
    "eta":         {"isim": 9,  "label": "#eta Dalitz"},
    "rho":         {"isim": 10, "label": "#rho #rightarrow e^{+}e^{-}"},
    "etap":        {"isim": 11, "label": "#eta' Dalitz"},
    "psip":        {"isim": 12, "label": "#psi(2S)"},
}

KEFFS = np.array([
    1.0, 1.3, 1.0, 1.3,
    25.e6 / 5.1e10 * 0.068**2 / 10.0 * 42.4 / 30.0,
    0.001102 * 10000 / 2104191 / 42 / 10 * 2,
    2.6, 1.0, 1.5e-7, 1.0, 1.3, 1.0, 1.0, 1.0, 1.0,
])

DNDY_BASE = np.array([
    95.0 / 257.0 * 42.2,
    0.421,
    0.421,
    0.759 / 1000.0,
    0.12,
    0.01,
    3.65,
    3.65,
    10.0,
    11.0 / 257.0 * 42.2,
    8.6 * 42.0 / 257.0,
    11.0 / 257.0 * 42.2,
    0.133 / 1000.0,
    1.0,
])

BR_TO_EE = np.array([
    1.174e-2, 2.97e-4, 1.08e-4, 5.94e-2,
    0.01, 0.01, 7.28e-5, 7.7e-4,
    1e-2, 6.9e-3, 4.73e-5, 4.73e-4,
    7.9e-3, 1.0,
])

DNDY_AUAU_MB = DNDY_BASE * NCOLL_AUAU_MB / 42.2
PI0_DNDY_AUAU_MB = float(DNDY_AUAU_MB[COMPONENTS["pi0"]["isim"]])

print("Au+Au 0-93% <Ncoll> =", NCOLL_AUAU_MB)
print("Au+Au pi0 dN/dy used for decay spectrum =", PI0_DNDY_AUAU_MB)
print("cocktail =", COCKTAIL_COMPONENTS)


Au+Au 0-93% <Ncoll> = 251.0
Au+Au pi0 dN/dy used for decay spectrum = 92.78210116731518
cocktail = ['pi0', 'eta']


In [5]:
# Histogram names copied from m_ee_decomposer74.

DATA_HISTOGRAMS = {
    "foreground": f"inv_mass_ee_DCA_V{DATA_OPTION}_FG12",
    "background": f"inv_mass_ee_DCA_V{DATA_OPTION}_BG12",
    "events": "PoolStatistics",
}

SIM_HISTOGRAM_NAMES = [
    "inv_mass_dca_fg0_6", "inv_mass_dca_fg0_5",
    "inv_mass_dca_fg0_6", "inv_mass_dca_fg0_7",
    "inv_mass_dca_fg0_8", "inv_mass_dca_fg0_9",
    "inv_mass_dca_fg2_6", "inv_mass_dca_fg2_5",
    "inv_mass_dca_fg2_6", "inv_mass_dca_fg2_7",
    "inv_mass_dca_fg2_8", "inv_mass_dca_fg2_9",
    "inv_mass_dca_fg4_9", "inv_mass_dca_fg4_5",
    "inv_mass_dca_fg4_6", "inv_mass_dca_fg4_7",
    "inv_mass_dca_fg4_8", "inv_mass_dca_fg4_9",
    "inv_mass_dca_gen_6", "inv_mass_dca_gen_5",
    "inv_mass_dca_gen_6", "inv_mass_dca_gen_7",
    "inv_mass_dca_gen_8", "inv_mass_dca_gen_9",
]

RECO_SHIFT = 12
GEN_SHIFT = 18
AUAU_CENTRALITY_SLICES = range(1, 6)
COCKTAIL_GLOBAL_SCALE = 0.8


In [6]:
def get_root_object(root_file, name):
    obj = root_file.Get(name)
    if not obj:
        raise KeyError(f"Missing ROOT object '{name}' in {root_file.GetName()}")
    obj.SetDirectory(0)
    return obj


# Load Au+Au data.
_data_file = root.TFile.Open(DATA_FILE, "READ")
if not _data_file or _data_file.IsZombie():
    raise OSError(f"Cannot open {DATA_FILE}")

h_data_fg = get_root_object(_data_file, DATA_HISTOGRAMS["foreground"])
h_data_bg = get_root_object(_data_file, DATA_HISTOGRAMS["background"])
h_event_count = get_root_object(_data_file, DATA_HISTOGRAMS["events"])
_data_file.Close()


# Load only the simulations needed by the selected cocktail plus eta.
components_to_load = set(COCKTAIL_COMPONENTS) | {DIRECT_TEMPLATE_SOURCE}
h_sim = {}

for name in sorted(components_to_load):
    path = SIM_FILES[name]
    infile = root.TFile.Open(path, "READ")
    if not infile or infile.IsZombie():
        raise OSError(f"Cannot open {path}")

    h_sim[name] = [
        get_root_object(infile, hist_name)
        for hist_name in SIM_HISTOGRAM_NAMES
    ]
    infile.Close()

print("Loaded data:", DATA_FILE)
print("Loaded simulations:", sorted(h_sim))


Loaded data: ../dca/input/data/m_ee_DCA_900.root
Loaded simulations: ['eta', 'pi0']


## Data and cocktail template helpers


In [7]:
def pt_bin_range(axis, low, high):
    first = axis.FindBin(low + 1e-9)
    last = axis.GetNbins() if high >= axis.GetXmax() else axis.FindBin(high - 1e-9)
    return first, last


def histogram_integral(hist, low, high, positive_only=False):
    total = 0.0
    for ibin in range(1, hist.GetNbinsX() + 1):
        x = hist.GetBinCenter(ibin)
        if not (low <= x < high):
            continue
        value = hist.GetBinContent(ibin)
        if positive_only:
            value = max(value, 0.0)
        total += value * hist.GetBinWidth(ibin)
    return total


def make_data_mass_histograms(pt_low, pt_high, dca_bin_low=1, dca_bin_high=None):
    if dca_bin_high is None:
        dca_bin_high = h_data_fg.GetYaxis().GetNbins()

    z1, z2 = pt_bin_range(h_data_fg.GetZaxis(), pt_low, pt_high)

    foreground = h_data_fg.ProjectionX(
        f"mass_fg_{pt_low}_{pt_high}", dca_bin_low, dca_bin_high, z1, z2
    )
    background = h_data_bg.ProjectionX(
        f"mass_bg_{pt_low}_{pt_high}", dca_bin_low, dca_bin_high, z1, z2
    )
    foreground.SetDirectory(0)
    background.SetDirectory(0)

    stat = foreground.Clone(f"mass_data_stat_{pt_low}_{pt_high}")
    syst = foreground.Clone(f"mass_data_syst_{pt_low}_{pt_high}")
    for hist in (stat, syst):
        hist.Reset("ICESM")
        hist.SetDirectory(0)

    n_events = max(float(h_event_count.GetBinContent(2)), 1.0)

    for ibin in range(1, foreground.GetNbinsX() + 1):
        width = foreground.GetBinWidth(ibin)
        fg = foreground.GetBinContent(ibin)
        bg = background.GetBinContent(ibin)

        value = (fg - SUBTRACTION_SCALE * bg) / (width * n_events)
        stat_error = np.sqrt(max(fg, 0.0)) / (width * n_events)
        syst_error = np.hypot(
            SUBTRACTION_SCALE * np.sqrt(max(bg, 0.0)),
            0.01 * SUBTRACTION_SCALE * abs(bg),
        ) / (width * n_events)

        stat.SetBinContent(ibin, value)
        stat.SetBinError(ibin, stat_error)
        syst.SetBinContent(ibin, value)
        syst.SetBinError(ibin, syst_error)

    return stat, syst


def make_component_mass_template(component, pt_low, pt_high):
    """Au+Au MB component normalization copied from m_ee_decomposer74."""
    isim = COMPONENTS[component]["isim"]
    histograms = h_sim[component]

    reference = histograms[RECO_SHIFT + 1]
    z1, z2 = pt_bin_range(reference.GetZaxis(), pt_low, pt_high)

    output = reference.ProjectionY(
        f"mass_{component}_{pt_low}_{pt_high}",
        1, reference.GetNbinsX(), z1, z2,
    )
    output.Reset("ICESM")
    output.SetDirectory(0)

    generated_entries = 0.0

    for centrality_slice in AUAU_CENTRALITY_SLICES:
        reco = histograms[RECO_SHIFT + centrality_slice]
        part = reco.ProjectionY(
            f"mass_{component}_{centrality_slice}_{pt_low}_{pt_high}",
            1, reco.GetNbinsX(), z1, z2,
        )
        part.SetDirectory(0)

        width = max(part.GetBinWidth(1), 1e-24)
        output.Add(
            part,
            10.0 / width * KEFFS[isim] * COCKTAIL_GLOBAL_SCALE,
        )

        generated_entries += histograms[GEN_SHIFT + centrality_slice].GetEntries()

    if generated_entries <= 0:
        raise RuntimeError(f"No generated entries for {component}")

    output.Scale(1.0 / generated_entries)
    return output


def add_histograms(histograms, name):
    if not histograms:
        raise ValueError("No histograms supplied.")
    result = histograms[0].Clone(name)
    result.SetDirectory(0)
    for hist in histograms[1:]:
        result.Add(hist)
    return result


## Direct virtual-photon template from reconstructed \(\eta\) Dalitz pairs


In [8]:
M_ELECTRON = 0.00051099895
M_ETA = 0.547862
ETA_POLE_MASS = 0.720
MAX_DIRECT_REWEIGHT = 50.0


def eta_specific_kroll_wada_factor(mass):
    if mass <= 2.0 * M_ELECTRON or mass >= M_ETA:
        return 0.0

    phase_space = max(1.0 - (mass / M_ETA)**2, 0.0)**3
    denominator = 1.0 - (mass / ETA_POLE_MASS)**2
    if denominator <= 0:
        return 0.0

    return phase_space / denominator**2


def direct_over_eta_weight(mass):
    eta_factor = eta_specific_kroll_wada_factor(mass)
    if eta_factor <= 0:
        return 0.0
    return min(1.0 / eta_factor, MAX_DIRECT_REWEIGHT)


def make_generated_component_mass_template(component, pt_low, pt_high):
    """
    Generated pair-mass spectrum summed over the five Au+Au centrality slices.

    The same component coefficient and generated-event normalization used for
    the reconstructed cocktail are applied here.
    """
    isim = COMPONENTS[component]["isim"]
    histograms = h_sim[component]

    output = None
    generated_entries = 0.0

    for centrality_slice in AUAU_CENTRALITY_SLICES:
        generated = histograms[GEN_SHIFT + centrality_slice]
        z1, z2 = pt_bin_range(generated.GetZaxis(), pt_low, pt_high)

        part = generated.ProjectionY(
            f"mass_gen_{component}_{centrality_slice}_{pt_low}_{pt_high}",
            1, generated.GetNbinsX(), z1, z2,
        )
        part.SetDirectory(0)

        if output is None:
            output = part.Clone(
                f"mass_gen_{component}_{pt_low}_{pt_high}"
            )
            output.Reset("ICESM")
            output.SetDirectory(0)

        width = max(part.GetBinWidth(1), 1e-24)
        output.Add(
            part,
            10.0 / width * KEFFS[isim] * COCKTAIL_GLOBAL_SCALE,
        )
        generated_entries += generated.GetEntries()

    if output is None or generated_entries <= 0:
        raise RuntimeError(
            f"No generated mass spectrum available for {component}."
        )

    output.Scale(1.0 / generated_entries)
    return output


def make_direct_mass_template(eta_reconstructed, eta_generated):
    """
    Reweight both reconstructed and generated eta Dalitz mass spectra to the
    direct virtual-photon Kroll-Wada shape.
    """
    weight_hist = eta_generated.Clone(
        eta_generated.GetName() + "_eta_to_direct_weight"
    )
    direct_generated = eta_generated.Clone(
        eta_generated.GetName() + "_direct"
    )

    weight_hist.Reset("ICESM")
    direct_generated.Reset("ICESM")
    weight_hist.SetDirectory(0)
    direct_generated.SetDirectory(0)

    for ibin in range(1, eta_generated.GetNbinsX() + 1):
        mass = eta_generated.GetBinCenter(ibin)
        weight = direct_over_eta_weight(mass)
        content = max(eta_generated.GetBinContent(ibin), 0.0)

        weight_hist.SetBinContent(ibin, weight)
        direct_generated.SetBinContent(ibin, content * weight)

    direct_reconstructed = eta_reconstructed.Clone(
        eta_reconstructed.GetName() + "_direct"
    )
    direct_reconstructed.Reset("ICESM")
    direct_reconstructed.SetDirectory(0)

    for ibin in range(1, direct_reconstructed.GetNbinsX() + 1):
        mass = direct_reconstructed.GetBinCenter(ibin)
        weight = weight_hist.GetBinContent(weight_hist.FindBin(mass))

        direct_reconstructed.SetBinContent(
            ibin,
            max(eta_reconstructed.GetBinContent(ibin), 0.0) * weight,
        )
        direct_reconstructed.SetBinError(
            ibin,
            eta_reconstructed.GetBinError(ibin) * weight,
        )

    return direct_reconstructed, direct_generated, weight_hist


def generated_ratio_correction(
    r_pair_fit,
    generated_cocktail,
    generated_direct,
    fit_range=MASS_FIT_RANGE,
    full_range=GENERATED_EXTRAPOLATION_RANGE,
):
    """
    Convert the fitted pair fraction to a generated full-range pair ratio.

    Let alpha be the direct-pair fraction in the fit window. Then

      O_fit = alpha / (1-alpha)

    and

      O_full = O_fit
               * (I_dir_full / I_dir_fit)
               / (I_cocktail_full / I_cocktail_fit).

    r_full = O_full / (1 + O_full)
    Rgamma_equivalent = 1 + O_full.

    This is a real-photon ratio only when generated direct and cocktail
    templates are normalized consistently per source photon.
    """
    alpha = float(np.clip(r_pair_fit, 0.0, 1.0 - 1e-12))

    cocktail_fit = histogram_integral(
        generated_cocktail, *fit_range, positive_only=True
    )
    direct_fit = histogram_integral(
        generated_direct, *fit_range, positive_only=True
    )
    cocktail_full = histogram_integral(
        generated_cocktail, *full_range, positive_only=True
    )
    direct_full = histogram_integral(
        generated_direct, *full_range, positive_only=True
    )

    values = (cocktail_fit, direct_fit, cocktail_full, direct_full)
    if any((not np.isfinite(v) or v <= 0) for v in values):
        raise RuntimeError(
            "Invalid generated integrals: "
            f"cocktail_fit={cocktail_fit:.3e}, "
            f"direct_fit={direct_fit:.3e}, "
            f"cocktail_full={cocktail_full:.3e}, "
            f"direct_full={direct_full:.3e}"
        )

    odds_fit = alpha / (1.0 - alpha)
    extrapolation = (
        (direct_full / direct_fit)
        / (cocktail_full / cocktail_fit)
    )
    odds_full = odds_fit * extrapolation

    return {
        "r_pair_fit": alpha,
        "pair_odds_fit": odds_fit,
        "generator_extrapolation_factor": extrapolation,
        "direct_to_decay_full": odds_full,
        "r_photon_equivalent": odds_full / (1.0 + odds_full),
        "Rgamma_equivalent": 1.0 + odds_full,
        "cocktail_fit_integral": cocktail_fit,
        "direct_fit_integral": direct_fit,
        "cocktail_full_integral": cocktail_full,
        "direct_full_integral": direct_full,
    }


## Mass fit with selectable cocktail and normalization mode


In [9]:
def fit_direct_fraction(
    data_hist,
    component_histograms,
    direct_hist,
    fit_range=MASS_FIT_RANGE,
    plot_range=MASS_PLOT_RANGE,
    lowmass_range=LOWMASS_RANGE,
    normalization_mode=COCKTAIL_NORMALIZATION_MODE,
):
    allowed_modes = {"original", "fixed_lowmass", "seed_from_lowmass"}
    if normalization_mode not in allowed_modes:
        raise ValueError(f"Unknown mode: {normalization_mode}")

    cocktail_hist = add_histograms(
        list(component_histograms.values()),
        data_hist.GetName() + "_cocktail",
    )

    y, sigma, cocktail_raw, direct_raw = [], [], [], []
    fit_low, fit_high = fit_range

    for ibin in range(1, data_hist.GetNbinsX() + 1):
        mass = data_hist.GetBinCenter(ibin)
        if not (fit_low <= mass < fit_high):
            continue

        width = data_hist.GetBinWidth(ibin)
        error = data_hist.GetBinError(ibin) * width
        if error <= 0 or not np.isfinite(error):
            continue

        y.append(data_hist.GetBinContent(ibin) * width)
        sigma.append(error)
        cocktail_raw.append(
            max(cocktail_hist.GetBinContent(cocktail_hist.FindBin(mass)) * width, 0.0)
        )
        direct_raw.append(
            max(direct_hist.GetBinContent(direct_hist.FindBin(mass)) * width, 0.0)
        )

    y = np.asarray(y, float)
    sigma = np.asarray(sigma, float)
    cocktail_raw = np.asarray(cocktail_raw, float)
    direct_raw = np.asarray(direct_raw, float)

    if len(y) < 2:
        raise RuntimeError("Too few valid mass bins for the fit.")

    low, high = lowmass_range
    data_low = histogram_integral(data_hist, low, high, True)
    cocktail_low = histogram_integral(cocktail_hist, low, high, True)
    direct_low = histogram_integral(direct_hist, low, high, True)

    lowmass_valid = all(
        np.isfinite(v) and v > 0
        for v in (data_low, cocktail_low, direct_low)
    )

    if normalization_mode != "original" and not lowmass_valid:
        raise RuntimeError(
            "Invalid low-mass normalization: "
            f"data={data_low:.3e}, cocktail={cocktail_low:.3e}, "
            f"direct={direct_low:.3e}"
        )

    direct_lowmass_scale = (
        cocktail_low / direct_low if lowmass_valid else 1.0
    )
    direct_matched = direct_raw * direct_lowmass_scale

    normalization_error = np.nan
    fit_success = True

    if normalization_mode == "original":
        cocktail_sum = cocktail_raw.sum()
        direct_sum = direct_raw.sum()
        normalization = np.maximum(y, 0.0).sum()

        if min(cocktail_sum, direct_sum, normalization) <= 0:
            raise RuntimeError("Non-positive fit integral.")

        cocktail_shape = cocktail_raw / cocktail_sum
        direct_shape = direct_raw / direct_sum

        base = normalization * cocktail_shape
        delta = normalization * (direct_shape - cocktail_shape)

        weights = 1.0 / sigma**2
        denominator = np.sum(weights * delta**2)
        numerator = np.sum(weights * delta * (y - base))

        r = float(np.clip(numerator / denominator, 0.0, 1.0))
        r_error = float(np.sqrt(1.0 / denominator))
        prediction = base + r * delta
        n_parameters = 1

        cocktail_draw_scale = normalization / cocktail_sum
        direct_draw_scale = normalization / direct_sum

    elif normalization_mode == "fixed_lowmass":
        normalization = data_low / cocktail_low
        base = normalization * cocktail_raw
        delta = normalization * (direct_matched - cocktail_raw)

        weights = 1.0 / sigma**2
        denominator = np.sum(weights * delta**2)
        numerator = np.sum(weights * delta * (y - base))

        r = float(np.clip(numerator / denominator, 0.0, 1.0))
        r_error = float(np.sqrt(1.0 / denominator))
        prediction = base + r * delta
        n_parameters = 1

        cocktail_draw_scale = normalization
        direct_draw_scale = normalization * direct_lowmass_scale

    else:
        normalization_seed = data_low / cocktail_low

        def residual(parameters):
            norm, fraction = parameters
            prediction = norm * (
                (1.0 - fraction) * cocktail_raw
                + fraction * direct_matched
            )
            return (y - prediction) / sigma

        fit = least_squares(
            residual,
            x0=[normalization_seed, 0.05],
            bounds=([0.0, 0.0], [np.inf, 1.0]),
            x_scale=[max(normalization_seed, 1e-20), 0.1],
            max_nfev=5000,
        )

        normalization, r = map(float, fit.x)
        fit_success = bool(fit.success)
        prediction = normalization * (
            (1.0 - r) * cocktail_raw + r * direct_matched
        )
        n_parameters = 2

        r_error = np.nan
        if fit.jac.shape[0] > fit.jac.shape[1]:
            try:
                covariance = np.linalg.inv(fit.jac.T @ fit.jac)
                normalization_error = float(np.sqrt(max(covariance[0, 0], 0.0)))
                r_error = float(np.sqrt(max(covariance[1, 1], 0.0)))
            except np.linalg.LinAlgError:
                pass

        cocktail_draw_scale = normalization
        direct_draw_scale = normalization * direct_lowmass_scale

    chi2 = float(np.sum(((y - prediction) / sigma)**2))
    ndf = max(len(y) - n_parameters, 0)

    # Draw-ready component histograms.
    component_outputs = {}
    for name, source in component_histograms.items():
        output = source.Clone(data_hist.GetName() + f"_fit_{name}")
        output.Reset("ICESM")
        output.SetDirectory(0)
        component_outputs[name] = output

    direct_output = direct_hist.Clone(data_hist.GetName() + "_fit_direct")
    cocktail_output = cocktail_hist.Clone(data_hist.GetName() + "_fit_cocktail")
    total_output = cocktail_hist.Clone(data_hist.GetName() + "_fit_total")

    for hist in (direct_output, cocktail_output, total_output):
        hist.Reset("ICESM")
        hist.SetDirectory(0)

    plot_low, plot_high = plot_range
    for ibin in range(1, data_hist.GetNbinsX() + 1):
        mass = data_hist.GetBinCenter(ibin)
        if not (plot_low <= mass < plot_high):
            continue

        cocktail_value = 0.0
        for name, source in component_histograms.items():
            raw = max(source.GetBinContent(source.FindBin(mass)), 0.0)
            value = (1.0 - r) * cocktail_draw_scale * raw
            component_outputs[name].SetBinContent(ibin, value)
            cocktail_value += value

        raw_direct = max(direct_hist.GetBinContent(direct_hist.FindBin(mass)), 0.0)
        direct_value = r * direct_draw_scale * raw_direct

        direct_output.SetBinContent(ibin, direct_value)
        cocktail_output.SetBinContent(ibin, cocktail_value)
        total_output.SetBinContent(ibin, cocktail_value + direct_value)

    return {
        "r": r,
        "r_err": r_error,
        "normalization": normalization,
        "normalization_err": normalization_error,
        "normalization_mode": normalization_mode,
        "fit_success": fit_success,
        "chi2": chi2,
        "ndf": ndf,
        "chi2_ndf": chi2 / ndf if ndf else np.nan,
        "lowmass_data": data_low,
        "lowmass_cocktail": cocktail_low,
        "direct_lowmass_scale": direct_lowmass_scale,
        "h_components": component_outputs,
        "h_cocktail": cocktail_output,
        "h_direct": direct_output,
        "h_total": total_output,
    }


In [10]:
_root_keepalive = {}
results = []

n_pt_bins = len(PT_BINS)
n_columns = min(3, max(1, math.ceil(math.sqrt(n_pt_bins))))
n_rows = math.ceil(n_pt_bins / n_columns)

canvas_mass = root.TCanvas(
    "c_mass_fits_AuAu_MB",
    "AuAu mass fits",
    440 * n_columns,
    380 * n_rows,
)
canvas_mass.Divide(n_columns, n_rows)
mass_objects = []

component_colors = {
    "pi0": root.kBlue + 1,
    "eta": root.kGreen + 2,
    "omega_ee": root.kOrange + 7,
    "omega_pi0ee": root.kCyan + 2,
    "rho": root.kViolet + 1,
    "etap": root.kGray + 2,
    "phi_etaee": root.kOrange + 2,
    "phi": root.kSpring + 4,
}

print("cocktail:", COCKTAIL_COMPONENTS)
print("normalization:", COCKTAIL_NORMALIZATION_MODE)
print("fit mass range:", MASS_FIT_RANGE)
print("generated extrapolation range:", GENERATED_EXTRAPOLATION_RANGE)
print("pT bin              r_ee(fit)        chi2/ndf")

for ipad, (pt_low, pt_high) in enumerate(PT_BINS, start=1):
    data_stat, data_syst = make_data_mass_histograms(pt_low, pt_high)

    component_templates = {
        name: make_component_mass_template(name, pt_low, pt_high)
        for name in COCKTAIL_COMPONENTS
    }
    generated_components = {
        name: make_generated_component_mass_template(name, pt_low, pt_high)
        for name in COCKTAIL_COMPONENTS
    }

    eta_reconstructed = (
        component_templates["eta"]
        if "eta" in component_templates
        else make_component_mass_template("eta", pt_low, pt_high)
    )
    eta_generated = (
        generated_components["eta"]
        if "eta" in generated_components
        else make_generated_component_mass_template("eta", pt_low, pt_high)
    )

    direct_template, direct_generated, weight_hist = (
        make_direct_mass_template(eta_reconstructed, eta_generated)
    )
    generated_cocktail = add_histograms(
        list(generated_components.values()),
        f"generated_cocktail_{pt_low}_{pt_high}",
    )

    fit_stat = fit_direct_fraction(
        data_stat, component_templates, direct_template
    )
    fit_syst = fit_direct_fraction(
        data_syst, component_templates, direct_template
    )

    generator_conversion = generated_ratio_correction(
        fit_stat["r"],
        generated_cocktail,
        direct_generated,
    )

    results.append({
        "pt_lo": pt_low,
        "pt_hi": pt_high,
        "r_stat": fit_stat["r"],
        "e_stat": fit_stat["r_err"],
        "r_syst": fit_syst["r"],
        "e_syst": fit_syst["r_err"],
        "fit_stat": fit_stat,
        "fit_syst": fit_syst,
        "generator_conversion": generator_conversion,
        "h_data": data_stat,
        "h_direct_template": direct_template,
        "h_generated_direct": direct_generated,
        "h_generated_cocktail": generated_cocktail,
        "component_templates": component_templates,
        "generated_components": generated_components,
    })

    print(
        f"{pt_low:4.2f}-{pt_high:4.2f}  "
        f"{fit_stat['r']:.4f} +/- {fit_stat['r_err']:.4f}  "
        f"{fit_stat['chi2']:.1f}/{fit_stat['ndf']}"
    )

    pad = canvas_mass.cd(ipad)
    pad.SetLogy()
    pad.SetTicks(1, 1)

    data_stat.SetStats(0)
    data_stat.SetMarkerStyle(20)
    data_stat.SetMarkerSize(0.7)
    data_stat.SetTitle(
        f"{pt_low:.2f} < p_{{T}} < {pt_high:.2f} GeV/c;"
        "m_{ee} (GeV/c^{2});1/N_{evt} dN/dm"
    )
    data_stat.GetXaxis().SetRangeUser(*MASS_PLOT_RANGE)
    data_stat.Draw("E1")

    drawn_components = {}
    for index, name in enumerate(COCKTAIL_COMPONENTS):
        hist = fit_stat["h_components"][name]
        hist.SetLineColor(component_colors.get(name, index + 2))
        hist.SetLineStyle(2 + index % 4)
        hist.SetLineWidth(2)
        hist.Draw("HIST SAME")
        drawn_components[name] = hist

    fit_stat["h_direct"].SetLineColor(root.kMagenta + 1)
    fit_stat["h_direct"].SetLineStyle(4)
    fit_stat["h_direct"].SetLineWidth(3)
    fit_stat["h_direct"].Draw("HIST SAME")

    fit_stat["h_total"].SetLineColor(root.kRed + 1)
    fit_stat["h_total"].SetLineWidth(3)
    fit_stat["h_total"].Draw("HIST SAME")
    data_stat.Draw("E1 SAME")

    legend = root.TLegend(0.44, 0.45, 0.89, 0.89)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)
    legend.SetTextSize(0.025)
    legend.AddEntry(data_stat, "Data", "lep")
    for name in COCKTAIL_COMPONENTS:
        legend.AddEntry(
            drawn_components[name], COMPONENTS[name]["label"], "l"
        )
    legend.AddEntry(fit_stat["h_direct"], "direct virtual #gamma", "l")
    legend.AddEntry(fit_stat["h_total"], "total fit", "l")
    legend.AddEntry(
        0,
        f"r_{{ee}}^{{fit}} = {fit_stat['r']:.3f} #pm {fit_stat['r_err']:.3f}",
        "",
    )
    legend.AddEntry(
        0,
        f"r_{{#gamma}}^{{equiv}} = "
        f"{generator_conversion['r_photon_equivalent']:.3f}",
        "",
    )
    legend.AddEntry(
        0,
        f"#chi^{{2}}/ndf = {fit_stat['chi2']:.1f}/{fit_stat['ndf']}",
        "",
    )
    legend.Draw()

    mass_objects.append({
        "data": data_stat,
        "components": drawn_components,
        "direct": fit_stat["h_direct"],
        "total": fit_stat["h_total"],
        "legend": legend,
        "generated_cocktail": generated_cocktail,
        "generated_direct": direct_generated,
        "weight": weight_hist,
    })

canvas_mass.Draw()

# Fitted dilepton fraction versus pT.
canvas_r = root.TCanvas("c_r_ee_AuAu_MB", "r_ee fit", 850, 620)
x = np.array([(row["pt_lo"] + row["pt_hi"]) / 2 for row in results], "float64")
ex = np.array([(row["pt_hi"] - row["pt_lo"]) / 2 for row in results], "float64")
y = np.array([row["r_stat"] for row in results], "float64")
ey = np.array([row["e_stat"] for row in results], "float64")

x_min = min(low for low, _ in PT_BINS)
x_max = max(high for _, high in PT_BINS)

frame_r = root.TH1D(
    "frame_r_ee_AuAu_MB",
    ";p_{T} (GeV/c);r_{ee}^{fit}",
    100, x_min, x_max,
)
frame_r.SetDirectory(0)
frame_r.SetStats(0)
frame_r.SetMinimum(-0.05)
frame_r.SetMaximum(max(1.05, 1.20 * float(np.max(y + ey))))
frame_r.Draw()

graph_r = root.TGraphErrors(len(x), x, y, ex, ey)
graph_r.SetMarkerStyle(20)
graph_r.Draw("P SAME")
canvas_r.Draw()

_root_keepalive["mass_fit"] = {
    "canvas": canvas_mass,
    "objects": mass_objects,
    "r_canvas": canvas_r,
    "r_frame": frame_r,
    "r_graph": graph_r,
}


cocktail: ['pi0', 'eta']
normalization: original
fit mass range: (0.03, 0.4)
generated extrapolation range: (0.0010219979, 0.5)
pT bin              r_ee(fit)        chi2/ndf
0.80-1.20  0.1713 +/- 0.0205  153.3/14
1.20-1.60  0.1819 +/- 0.0203  126.1/14
1.60-2.00  0.2492 +/- 0.0230  40.0/14
2.00-2.80  0.2435 +/- 0.0306  58.0/14
2.80-3.60  0.3826 +/- 0.0530  20.9/14
3.60-4.20  0.3715 +/- 0.0811  24.5/13
4.20-6.00  0.3090 +/- 0.0817  41.0/13


## Convert \(r\) to \(R_\gamma\)

\[
R_\gamma=\frac{1}{1-r}.
\]


In [11]:
def convert_with_uncertainty(result, r_value):
    return generated_ratio_correction(
        r_value,
        result["h_generated_cocktail"],
        result["h_generated_direct"],
    )


rows = []

for result in results:
    r_fit = result["r_stat"]
    r_fit_error = max(float(result["e_stat"]), 0.0)

    central = convert_with_uncertainty(result, r_fit)
    low = convert_with_uncertainty(
        result, max(0.0, r_fit - r_fit_error)
    )
    high = convert_with_uncertainty(
        result, min(1.0 - 1e-9, r_fit + r_fit_error)
    )

    r_syst_error = float(np.hypot(
        result["r_syst"] - result["r_stat"],
        result["e_syst"],
    ))
    syst_low = convert_with_uncertainty(
        result, max(0.0, r_fit - r_syst_error)
    )
    syst_high = convert_with_uncertainty(
        result, min(1.0 - 1e-9, r_fit + r_syst_error)
    )

    rows.append({
        "pt_low": result["pt_lo"],
        "pt_high": result["pt_hi"],
        "pt": 0.5 * (result["pt_lo"] + result["pt_hi"]),
        "pt_err": 0.5 * (result["pt_hi"] - result["pt_lo"]),

        # Direct output of the dilepton shape fit.
        "r_ee_fit": r_fit,
        "r_ee_fit_stat": r_fit_error,

        # Generator extrapolation information.
        "generator_extrapolation_factor":
            central["generator_extrapolation_factor"],
        "direct_to_decay": central["direct_to_decay_full"],

        # Photon-equivalent fraction and Rgamma.
        "r": central["r_photon_equivalent"],
        "r_stat_down":
            central["r_photon_equivalent"] - low["r_photon_equivalent"],
        "r_stat_up":
            high["r_photon_equivalent"] - central["r_photon_equivalent"],

        "Rgamma": central["Rgamma_equivalent"],
        "Rgamma_stat_down":
            central["Rgamma_equivalent"] - low["Rgamma_equivalent"],
        "Rgamma_stat_up":
            high["Rgamma_equivalent"] - central["Rgamma_equivalent"],
        "Rgamma_syst_down":
            central["Rgamma_equivalent"] - syst_low["Rgamma_equivalent"],
        "Rgamma_syst_up":
            syst_high["Rgamma_equivalent"] - central["Rgamma_equivalent"],

        "generated_cocktail_fit_integral":
            central["cocktail_fit_integral"],
        "generated_direct_fit_integral":
            central["direct_fit_integral"],
        "generated_cocktail_full_integral":
            central["cocktail_full_integral"],
        "generated_direct_full_integral":
            central["direct_full_integral"],
    })

df_rgamma = pd.DataFrame(rows)

display(df_rgamma[[
    "pt_low",
    "pt_high",
    "r_ee_fit",
    "generator_extrapolation_factor",
    "direct_to_decay",
    "r",
    "Rgamma",
]])


,pt_low,pt_high,r_ee_fit,generator_extrapolation_factor,direct_to_decay,r,Rgamma
0,0.8,1.2,0.171341,0.472829,0.097767,0.089060,1.097767
1,1.2,1.6,0.181878,0.489106,0.108734,0.098071,1.108734
2,1.6,2.0,0.249212,0.491107,0.163015,0.140166,1.163015
3,2.0,2.8,0.243456,0.493965,0.158958,0.137156,1.158958
4,2.8,3.6,0.382623,0.496000,0.307399,0.235122,1.307399
5,3.6,4.2,0.371519,0.499911,0.295517,0.228107,1.295517
6,4.2,6.0,0.309045,0.500620,0.223913,0.182949,1.223913


## Self-contained Au+Au decay-photon spectrum


In [12]:
# Hagedorn shape from the previous direct-photon helper notebook.
# It is renormalized below to the Au+Au 0-93% pi0 dN/dy from m_ee_decomposer74.

PI0_HAGEDORN = {
    "A": 1.922170e2,
    "p0": 2.106180,
    "n": 12.84030,
    "B": 16.30000,
    "m": 8.060480,
    "transition": 4.033000,
    "width": 0.06395340,
}

M_PI0 = 0.1349768


def raw_pi0_shape(pt):
    pt = np.asarray(pt, float)
    p = PI0_HAGEDORN
    switch = 1.0 / (
        1.0 + np.exp(
            np.clip((pt - p["transition"]) / p["width"], -700, 700)
        )
    )
    low = p["A"] / (1.0 + pt / p["p0"])**p["n"]
    high = np.zeros_like(pt)
    mask = pt > 0
    high[mask] = p["B"] / pt[mask]**p["m"]
    return switch * low + (1.0 - switch) * high


_normalization_grid = np.linspace(1e-4, 20.0, 200000)
_raw_dndy = np.trapz(
    2.0 * np.pi * _normalization_grid * raw_pi0_shape(_normalization_grid),
    _normalization_grid,
)
PI0_AUAU_NORMALIZATION = PI0_DNDY_AUAU_MB / _raw_dndy


def pi0_invariant_yield_auau(pt):
    return PI0_AUAU_NORMALIZATION * raw_pi0_shape(pt)


def pi0_dndpt_auau(pt):
    pt = np.asarray(pt, float)
    return 2.0 * np.pi * pt * pi0_invariant_yield_auau(pt)


print("raw shape dN/dy =", _raw_dndy)
print("Au+Au normalization factor =", PI0_AUAU_NORMALIZATION)


raw shape dN/dy = 41.74107871379739
Au+Au normalization factor = 2.2228007523113273


In [13]:
# Full hadronic decay photons / pi0-decay photons.
_RATIO_PT = np.array([
    0.55, 0.60, 0.70, 0.80, 0.90, 1.00,
    1.25, 1.50, 1.75, 2.00, 2.25, 2.50,
    2.75, 3.00, 3.50, 4.00, 4.50, 5.00,
    5.50, 6.00, 6.50, 7.00, 7.50, 8.00,
    8.50, 9.00, 9.50, 9.90,
])

_RATIO_VALUE = np.array([
    1.09696, 1.11315, 1.13620, 1.14979, 1.15773, 1.16320,
    1.17504, 1.18419, 1.19127, 1.19712, 1.20184, 1.20584,
    1.20926, 1.21221, 1.21708, 1.22092, 1.22365, 1.22579,
    1.22739, 1.22838, 1.22922, 1.22976, 1.22995, 1.23024,
    1.23024, 1.23028, 1.22996, 1.22980,
])


def hadron_decay_over_pi0_decay(pt):
    return np.interp(
        np.asarray(pt, float),
        _RATIO_PT,
        _RATIO_VALUE,
        left=_RATIO_VALUE[0],
        right=_RATIO_VALUE[-1],
    )


In [14]:
def generate_pi0_decay_spectrum(
    pt_edges,
    n_mc=3_000_000,
    chunk_size=250_000,
    parent_pt_max=20.0,
    seed=12345,
):
    rng = np.random.default_rng(seed)
    pt_edges = np.asarray(pt_edges, float)

    parent_hist = np.zeros(len(pt_edges) - 1)
    photon_hist = np.zeros(len(pt_edges) - 1)

    generated = 0
    while generated < n_mc:
        size = min(chunk_size, n_mc - generated)

        parent_pt = rng.uniform(0.0, parent_pt_max, size)
        weight = pi0_dndpt_auau(parent_pt)

        parent_energy = np.sqrt(parent_pt**2 + M_PI0**2)
        beta = parent_pt / parent_energy
        gamma = parent_energy / M_PI0

        cos_theta = rng.uniform(-1.0, 1.0, size)
        sin_theta = np.sqrt(np.maximum(1.0 - cos_theta**2, 0.0))
        phi = rng.uniform(0.0, 2.0 * np.pi, size)

        p_star = M_PI0 / 2.0
        px_star = p_star * sin_theta * np.cos(phi)
        py_star = p_star * sin_theta * np.sin(phi)

        photon1_pt = np.hypot(
            gamma * (px_star + beta * p_star),
            py_star,
        )
        photon2_pt = np.hypot(
            gamma * (-px_star + beta * p_star),
            -py_star,
        )

        parent_hist += np.histogram(
            parent_pt, bins=pt_edges, weights=weight
        )[0]
        photon_hist += np.histogram(
            photon1_pt, bins=pt_edges, weights=weight
        )[0]
        photon_hist += np.histogram(
            photon2_pt, bins=pt_edges, weights=weight
        )[0]

        generated += size

    mc_norm = parent_pt_max / n_mc
    centers = 0.5 * (pt_edges[:-1] + pt_edges[1:])
    widths = np.diff(pt_edges)
    invariant_denominator = 2.0 * np.pi * centers * widths

    parent_inv = mc_norm * parent_hist / invariant_denominator
    photon_inv = mc_norm * photon_hist / invariant_denominator
    parent_input = pi0_invariant_yield_auau(centers)

    return pd.DataFrame({
        "pt_low": pt_edges[:-1],
        "pt_high": pt_edges[1:],
        "pt": centers,
        "pi0_parent_input": parent_input,
        "pi0_parent_mc": parent_inv,
        "gamma_pi0_decay": photon_inv,
        "gamma_pi0_decay_over_pi0_parent": np.divide(
            photon_inv,
            parent_input,
            out=np.full_like(photon_inv, np.nan),
            where=parent_input > 0,
        ),
    })


pt_edges_decay = np.unique(np.r_[
    df_rgamma["pt_low"].to_numpy(float),
    float(df_rgamma["pt_high"].iloc[-1]),
])

df_decay_gamma = generate_pi0_decay_spectrum(pt_edges_decay)
df_decay_gamma["hadron_over_pi0_decay"] = hadron_decay_over_pi0_decay(
    df_decay_gamma["pt"]
)
df_decay_gamma["gamma_decay"] = (
    df_decay_gamma["gamma_pi0_decay"]
    * df_decay_gamma["hadron_over_pi0_decay"]
)

# No independent uncertainties for these inputs were provided in the source notebook.
df_decay_gamma["gamma_decay_stat"] = 0.0
df_decay_gamma["gamma_decay_syst_down"] = 0.0
df_decay_gamma["gamma_decay_syst_up"] = 0.0

display(df_decay_gamma)


,pt_low,pt_high,pt,pi0_parent_input,pi0_parent_mc,gamma_pi0_decay,gamma_pi0_decay_over_pi0_parent,hadron_over_pi0_decay,gamma_decay,gamma_decay_stat,gamma_decay_syst_down,gamma_decay_syst_up
0,0.8,1.2,1.0,2.911615,3.121223,1.723539,0.591953,1.163200,2.004821,0.0,0.0,0.0
1,1.2,1.6,1.4,0.614662,0.656123,0.285204,0.464002,1.180530,0.336692,0.0,0.0,0.0
2,1.6,2.0,1.8,0.153531,0.162457,0.060388,0.393328,1.192440,0.072009,0.0,0.0,0.0
3,2.0,2.8,2.4,0.024512,0.028968,0.009593,0.391370,1.204240,0.011552,0.0,0.0,0.0
4,2.8,3.6,3.2,0.003006,0.003442,0.000995,0.331105,1.214158,0.001209,0.0,0.0,0.0
5,3.6,4.2,3.9,0.000614,0.000653,0.000188,0.305740,1.220152,0.000229,0.0,0.0,0.0
6,4.2,6.0,5.1,0.000072,0.000096,0.000027,0.379741,1.226110,0.000033,0.0,0.0,0.0


## Real direct-photon spectrum


In [15]:
import numpy as np
import pandas as pd

if "df_decay_gamma" not in globals():
    raise RuntimeError(
        "Create df_decay_gamma first. Required columns: "
        "pt_low, pt_high, gamma_decay."
    )

required = {"pt_low", "pt_high", "gamma_decay"}
missing = required - set(df_decay_gamma.columns)
if missing:
    raise KeyError(f"df_decay_gamma is missing: {sorted(missing)}")

df_direct_gamma = df_rgamma.merge(
    df_decay_gamma,
    on=["pt_low", "pt_high"],
    how="inner",
    validate="one_to_one",
    suffixes=("", "_decay"),
)

df_direct_gamma["pt"] = 0.5 * (
    df_direct_gamma["pt_low"] + df_direct_gamma["pt_high"]
)
df_direct_gamma["pt_err"] = 0.5 * (
    df_direct_gamma["pt_high"] - df_direct_gamma["pt_low"]
)

for name in (
    "gamma_decay_stat",
    "gamma_decay_syst_down",
    "gamma_decay_syst_up",
):
    if name not in df_direct_gamma:
        df_direct_gamma[name] = 0.0

G = df_direct_gamma["gamma_decay"].to_numpy(float)
R = df_direct_gamma["Rgamma"].to_numpy(float)
df_direct_gamma["gamma_direct"] = (R - 1.0) * G

def combine(a, b):
    return np.hypot(np.asarray(a, float), np.asarray(b, float))

rstat_down = df_direct_gamma.get(
    "Rgamma_stat_down", df_direct_gamma.get("Rgamma_stat", 0.0)
)
rstat_up = df_direct_gamma.get(
    "Rgamma_stat_up", df_direct_gamma.get("Rgamma_stat", 0.0)
)
rsyst_down = df_direct_gamma.get("Rgamma_syst_down", 0.0)
rsyst_up = df_direct_gamma.get("Rgamma_syst_up", 0.0)

Gstat = df_direct_gamma["gamma_decay_stat"]
Gsyst_down = df_direct_gamma["gamma_decay_syst_down"]
Gsyst_up = df_direct_gamma["gamma_decay_syst_up"]

df_direct_gamma["gamma_direct_stat_down"] = combine(
    G * rstat_down, (R - 1.0) * Gstat
)
df_direct_gamma["gamma_direct_stat_up"] = combine(
    G * rstat_up, (R - 1.0) * Gstat
)
df_direct_gamma["gamma_direct_stat"] = 0.5 * (
    df_direct_gamma["gamma_direct_stat_down"]
    + df_direct_gamma["gamma_direct_stat_up"]
)
df_direct_gamma["gamma_direct_syst_down"] = combine(
    G * rsyst_down, (R - 1.0) * Gsyst_down
)
df_direct_gamma["gamma_direct_syst_up"] = combine(
    G * rsyst_up, (R - 1.0) * Gsyst_up
)

display(df_direct_gamma[
    [
        "pt_low", "pt_high", "Rgamma",
        "gamma_decay", "gamma_direct",
        "gamma_direct_stat",
        "gamma_direct_syst_down",
        "gamma_direct_syst_up",
    ]
])


,pt_low,pt_high,Rgamma,gamma_decay,gamma_direct,gamma_direct_stat,gamma_direct_syst_down,gamma_direct_syst_up
0,0.8,1.2,1.097767,2.004821,0.196005,0.028322,0.063025,0.070829
1,1.2,1.6,1.108734,0.336692,0.036610,0.005000,0.009979,0.011077
2,1.6,2.0,1.163015,0.072009,0.011739,0.001445,0.000183,0.000184
3,2.0,2.8,1.158958,0.011552,0.001836,0.000306,0.000067,0.000069
4,2.8,3.6,1.307399,0.001209,0.000372,0.000084,0.000151,0.000219
5,3.6,4.2,1.295517,0.000229,0.000068,0.000024,0.000068,0.000384
6,4.2,6.0,1.223913,0.000033,0.000007,0.000003,0.000007,16720.839365


## DCA closure of the final mass-fit result


In [16]:
def make_data_dca_histogram(pt_low, pt_high, mass_range=MASS_FIT_RANGE):
    x1 = h_data_fg.GetXaxis().FindBin(mass_range[0] + 1e-9)
    x2 = h_data_fg.GetXaxis().FindBin(mass_range[1] - 1e-9)
    z1, z2 = pt_bin_range(h_data_fg.GetZaxis(), pt_low, pt_high)

    fg = h_data_fg.ProjectionY(f"dca_fg_{pt_low}_{pt_high}", x1, x2, z1, z2)
    bg = h_data_bg.ProjectionY(f"dca_bg_{pt_low}_{pt_high}", x1, x2, z1, z2)
    fg.SetDirectory(0)
    bg.SetDirectory(0)

    output = fg.Clone(f"dca_data_{pt_low}_{pt_high}")
    output.Reset("ICESM")
    output.SetDirectory(0)

    n_events = max(float(h_event_count.GetBinContent(2)), 1.0)

    for ibin in range(1, output.GetNbinsX() + 1):
        width = output.GetBinWidth(ibin)
        n_fg = fg.GetBinContent(ibin)
        n_bg = bg.GetBinContent(ibin)

        output.SetBinContent(
            ibin,
            (n_fg - SUBTRACTION_SCALE * n_bg) / (n_events * width),
        )
        output.SetBinError(
            ibin,
            np.sqrt(max(n_fg, 0.0) + SUBTRACTION_SCALE**2 * max(n_bg, 0.0))
            / (n_events * width),
        )

    return output


def make_component_dca_template(component, pt_low, pt_high,
                                mass_range=MASS_FIT_RANGE):
    isim = COMPONENTS[component]["isim"]
    histograms = h_sim[component]
    output = None
    generated_entries = 0.0

    for centrality_slice in AUAU_CENTRALITY_SLICES:
        reco = histograms[RECO_SHIFT + centrality_slice]
        y1 = reco.GetYaxis().FindBin(mass_range[0] + 1e-9)
        y2 = reco.GetYaxis().FindBin(mass_range[1] - 1e-9)
        z1, z2 = pt_bin_range(reco.GetZaxis(), pt_low, pt_high)

        part = reco.ProjectionX(
            f"dca_{component}_{centrality_slice}_{pt_low}_{pt_high}",
            y1, y2, z1, z2,
        )
        part.SetDirectory(0)

        if output is None:
            output = part.Clone(f"dca_{component}_{pt_low}_{pt_high}")
            output.Reset("ICESM")
            output.SetDirectory(0)

        width = max(part.GetBinWidth(1), 1e-24)
        output.Add(part, 10.0 / width * KEFFS[isim] * COCKTAIL_GLOBAL_SCALE)
        generated_entries += histograms[GEN_SHIFT + centrality_slice].GetEntries()

    output.Scale(1.0 / generated_entries)
    return output


def make_direct_dca_template(pt_low, pt_high, mass_range=MASS_FIT_RANGE):
    eta_histograms = h_sim[DIRECT_TEMPLATE_SOURCE]
    output = None

    for centrality_slice in AUAU_CENTRALITY_SLICES:
        reco = eta_histograms[RECO_SHIFT + centrality_slice]
        y1 = reco.GetYaxis().FindBin(mass_range[0] + 1e-9)
        y2 = reco.GetYaxis().FindBin(mass_range[1] - 1e-9)
        z1, z2 = pt_bin_range(reco.GetZaxis(), pt_low, pt_high)

        for iy in range(y1, y2 + 1):
            mass = reco.GetYaxis().GetBinCenter(iy)
            part = reco.ProjectionX(
                f"dca_direct_{centrality_slice}_{iy}_{pt_low}_{pt_high}",
                iy, iy, z1, z2,
            )
            part.SetDirectory(0)

            if output is None:
                output = part.Clone(f"dca_direct_{pt_low}_{pt_high}")
                output.Reset("ICESM")
                output.SetDirectory(0)

            output.Add(part, direct_over_eta_weight(mass))

    return output


def normalize_shape(hist, name):
    output = hist.Clone(name)
    output.SetDirectory(0)
    integral = output.Integral(1, output.GetNbinsX(), "width")
    if integral > 0:
        output.Scale(1.0 / integral)
    return output


def dca_chi2(data, model):
    chi2 = 0.0
    valid_bins = 0
    for ibin in range(1, data.GetNbinsX() + 1):
        error = data.GetBinError(ibin)
        if error <= 0:
            continue
        chi2 += (
            (data.GetBinContent(ibin) - model.GetBinContent(ibin)) / error
        )**2
        valid_bins += 1
    ndf = max(valid_bins - 1, 0)
    return chi2, ndf


In [17]:
_dca_keepalive = {}
dca_rows = []

n_pt_bins = len(PT_BINS)
n_columns = min(3, max(1, math.ceil(math.sqrt(n_pt_bins))))
n_rows = math.ceil(n_pt_bins / n_columns)

canvas_dca = root.TCanvas(
    "c_dca_AuAu_MB",
    "DCA closure",
    440 * n_columns,
    380 * n_rows,
)
canvas_dca.Divide(n_columns, n_rows)

for ipad, result in enumerate(results, start=1):
    pt_low, pt_high = result["pt_lo"], result["pt_hi"]
    fraction = result["r_stat"]

    data = make_data_dca_histogram(pt_low, pt_high)
    cocktail_parts = [
        make_component_dca_template(name, pt_low, pt_high)
        for name in COCKTAIL_COMPONENTS
    ]
    cocktail = add_histograms(
        cocktail_parts, f"dca_cocktail_{pt_low}_{pt_high}"
    )
    direct = make_direct_dca_template(pt_low, pt_high)

    cocktail = normalize_shape(cocktail, cocktail.GetName() + "_norm")
    direct = normalize_shape(direct, direct.GetName() + "_norm")

    model = cocktail.Clone(f"dca_model_{pt_low}_{pt_high}")
    model.Scale(1.0 - fraction)
    model.Add(direct, fraction)

    data_integral = data.Integral(1, data.GetNbinsX(), "width")
    model_integral = model.Integral(1, model.GetNbinsX(), "width")
    if model_integral > 0:
        model.Scale(data_integral / model_integral)

    chi2, ndf = dca_chi2(data, model)
    dca_rows.append({
        "pt_low": pt_low,
        "pt_high": pt_high,
        "r": fraction,
        "chi2": chi2,
        "ndf": ndf,
        "chi2_ndf": chi2 / ndf if ndf else np.nan,
    })

    pad = canvas_dca.cd(ipad)
    pad.SetLogy()
    pad.SetTicks(1, 1)

    data.SetStats(0)
    data.SetMarkerStyle(20)
    data.SetTitle(
        f"{pt_low:.1f} < p_{{T}} < {pt_high:.1f} GeV/c;"
        "pair DCA;yield"
    )
    data.Draw("E1")

    model.SetLineColor(root.kRed + 1)
    model.SetLineWidth(3)
    model.Draw("HIST SAME")
    data.Draw("E1 SAME")

    legend = root.TLegend(0.50, 0.74, 0.89, 0.89)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)
    legend.AddEntry(data, "data", "lep")
    legend.AddEntry(model, "mass-fit prediction", "l")
    legend.AddEntry(0, f"#chi^{{2}}/ndf = {chi2:.1f}/{ndf}", "")
    legend.Draw()

    _dca_keepalive[ipad] = {
        "data": data,
        "cocktail": cocktail,
        "direct": direct,
        "model": model,
        "legend": legend,
    }

canvas_dca.Draw()
_dca_keepalive["canvas"] = canvas_dca

df_dca_chi2 = pd.DataFrame(dca_rows)
display(df_dca_chi2)


,pt_low,pt_high,r,chi2,ndf,chi2_ndf
0,0.8,1.2,0.171341,491.675964,33,14.899272
1,1.2,1.6,0.181878,194.683697,33,5.899506
2,1.6,2.0,0.249212,139.867576,29,4.823020
3,2.0,2.8,0.243456,66.343551,28,2.369413
4,2.8,3.6,0.382623,33.588983,28,1.199607
5,3.6,4.2,0.371519,10.513467,18,0.584082
6,4.2,6.0,0.309045,13.642412,16,0.852651


## Final Au+Au results and \(N_{
m coll}\)-scaled p+p reference


In [18]:
# p+p direct-photon fit used previously, scaled by Au+Au 0-93% <Ncoll>.
PP_DIRECT_FIT = {
    "A": 1.60e-4,
    "p0": 1.45,
    "n": 3.3,
}


def pp_direct_reference(pt):
    pt = np.asarray(pt, float)
    p = PP_DIRECT_FIT
    pp = p["A"] / (1.0 + (pt / p["p0"])**2)**p["n"]
    return NCOLL_AUAU_MB * pp


_plot_keepalive = {}


In [21]:
# Rgamma
c_rgamma = root.TCanvas("c_rgamma_AuAu_MB", "Rgamma", 800, 600)
c_rgamma.SetTicks(1, 1)

xmin = float(df_rgamma["pt_low"].min())
xmax = float(df_rgamma["pt_high"].max())

frame = root.TH1D(
    "frame_rgamma_pp",
    ";p_{T} (GeV/c);R_{#gamma}",
    100, xmin, xmax,
)
frame.SetDirectory(0)
frame.SetStats(0)
frame.SetMinimum(0.8)
frame.SetMaximum(min(
    2.0,
    1.25 * float(
        (df_rgamma["Rgamma"] + df_rgamma["Rgamma_syst_up"]).max()
    ),
))
frame.Draw()

g_syst = root.TGraphAsymmErrors(len(df_rgamma))
g_stat = root.TGraphAsymmErrors(len(df_rgamma))

for i, row in enumerate(df_rgamma.itertuples(index=False)):
    g_syst.SetPoint(i, row.pt, row.Rgamma)
    g_syst.SetPointError(
        i, row.pt_err, row.pt_err,
        row.Rgamma_syst_down, row.Rgamma_syst_up,
    )
    g_stat.SetPoint(i, row.pt, row.Rgamma)
    g_stat.SetPointError(
        i, 0.0, 0.0,
        row.Rgamma_stat_down, row.Rgamma_stat_up,
    )

g_syst.SetFillColorAlpha(root.kAzure + 1, 0.30)
g_stat.SetMarkerStyle(20)
g_stat.SetMarkerSize(1.1)

g_syst.Draw("E2 SAME")
g_stat.Draw("P SAME")

unity = root.TLine(xmin, 1.0, xmax, 1.0)
unity.SetLineStyle(2)
unity.Draw()

label = root.TLatex()
label.SetNDC()
label.SetTextSize(0.04)
label.DrawLatex(0.16, 0.86, "0-93% Au+Au, #sqrt{s_{NN}} = 200 GeV")

c_rgamma.Draw()

_plot_keepalive["rgamma"] = (
    c_rgamma, frame, g_syst, g_stat, unity, label
)


In [20]:
# Direct-photon invariant yield and p+p reference
valid = (
    np.isfinite(df_direct_gamma["gamma_direct"])
    & (df_direct_gamma["gamma_direct"] > 0)
)
plot_data = df_direct_gamma.loc[valid].copy()

if plot_data.empty:
    raise RuntimeError("No positive direct-photon points to plot.")

pt_curve = np.linspace(
    max(0.8, float(plot_data["pt_low"].min())),
    float(plot_data["pt_high"].max()),
    300,
)
yield_curve = pp_direct_reference(pt_curve)

c_direct = root.TCanvas("c_direct_AuAu_MB", "Direct photons", 800, 600)
c_direct.SetTicks(1, 1)
c_direct.SetLogy()

ymin = min(
    float(plot_data["gamma_direct"].min()),
    float(yield_curve.min()),
) * 0.2
ymax = max(
    float(
        (
            plot_data["gamma_direct"]
            + plot_data["gamma_direct_syst_up"]
        ).max()
    ),
    float(yield_curve.max()),
) * 5.0

frame = root.TH1D(
    "frame_direct_pp",
    (
        ";p_{T} (GeV/c);"
        "#frac{1}{2#pip_{T}}"
        "#frac{d^{2}N_{#gamma}^{dir}}{dp_{T}dy}"
    ),
    100,
    float(plot_data["pt_low"].min()),
    float(plot_data["pt_high"].max()),
)
frame.SetDirectory(0)
frame.SetStats(0)
frame.SetMinimum(max(ymin, 1e-15))
frame.SetMaximum(ymax)
frame.Draw()

g_ref = root.TGraph(len(pt_curve))
for i, (pt, value) in enumerate(zip(pt_curve, yield_curve)):
    g_ref.SetPoint(i, float(pt), float(value))
g_ref.SetLineColor(root.kOrange + 7)
g_ref.SetLineWidth(3)
g_ref.Draw("L SAME")

g_syst = root.TGraphAsymmErrors(len(plot_data))
g_stat = root.TGraphErrors(len(plot_data))

for i, row in enumerate(plot_data.itertuples(index=False)):
    g_syst.SetPoint(i, row.pt, row.gamma_direct)
    g_syst.SetPointError(
        i, row.pt_err, row.pt_err,
        row.gamma_direct_syst_down,
        row.gamma_direct_syst_up,
    )
    g_stat.SetPoint(i, row.pt, row.gamma_direct)
    g_stat.SetPointError(i, 0.0, row.gamma_direct_stat)

g_syst.SetFillColorAlpha(root.kAzure + 1, 0.30)
g_stat.SetMarkerStyle(20)
g_stat.SetMarkerSize(1.1)

g_syst.Draw("E2 SAME")
g_stat.Draw("P SAME")

legend = root.TLegend(0.48, 0.70, 0.89, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(g_stat, "extracted direct #gamma", "p")
legend.AddEntry(g_syst, "systematic uncertainty", "f")
legend.AddEntry(g_ref, "#LTN_{coll}#GT-scaled p+p fit", "l")
legend.Draw()

label = root.TLatex()
label.SetNDC()
label.SetTextSize(0.04)
label.DrawLatex(0.16, 0.86, "0-93% Au+Au, #sqrt{s_{NN}} = 200 GeV")

c_direct.Draw()

_plot_keepalive["direct"] = (
    c_direct, frame, g_ref, g_syst, g_stat, legend, label
)


In [22]:
from pathlib import Path

RGAMMA_OUTPUT = (
    "output/final_arrays/"
    "auau_mb_rgamma_generator_corrected.csv"
)

Path(RGAMMA_OUTPUT).parent.mkdir(
    parents=True,
    exist_ok=True,
)

rgamma_columns = [
    "pt_low",
    "pt_high",
    "pt",
    "pt_err",

    # Dilepton fraction directly returned by the mass fit.
    "r_ee_fit",
    "r_ee_fit_stat",

    # Generator conversion.
    "generator_extrapolation_factor",
    "direct_to_decay",

    # Photon-equivalent quantities.
    "r",
    "r_stat_down",
    "r_stat_up",

    "Rgamma",
    "Rgamma_stat_down",
    "Rgamma_stat_up",
    "Rgamma_syst_down",
    "Rgamma_syst_up",

    # Useful diagnostics.
    "generated_cocktail_fit_integral",
    "generated_direct_fit_integral",
    "generated_cocktail_full_integral",
    "generated_direct_full_integral",
]

missing = [
    column
    for column in rgamma_columns
    if column not in df_rgamma.columns
]

if missing:
    raise KeyError(
        f"df_rgamma is missing columns: {missing}"
    )

df_rgamma[rgamma_columns].to_csv(
    RGAMMA_OUTPUT,
    index=False,
)

print("Saved:", RGAMMA_OUTPUT)

Saved: output/final_arrays/auau_mb_rgamma_generator_corrected.csv


In [23]:
DIRECT_OUTPUT = (
    "output/final_arrays/"
    "auau_mb_direct_gamma.csv"
)

Path(DIRECT_OUTPUT).parent.mkdir(
    parents=True,
    exist_ok=True,
)

direct_columns = [
    "pt_low",
    "pt_high",
    "pt",
    "pt_err",
    "Rgamma",
    "Rgamma_stat_down",
    "Rgamma_stat_up",
    "Rgamma_syst_down",
    "Rgamma_syst_up",
    "gamma_decay",
    "gamma_direct",
    "gamma_direct_stat",
    "gamma_direct_syst_down",
    "gamma_direct_syst_up",
]

# Add the approved Au+Au MB multiplicity before saving.
# Replace these with the values you intend to use.
DNCH_AUAU_MB = None
DNCH_AUAU_MB_ERR = None

if DNCH_AUAU_MB is not None:
    df_direct_gamma["dNch"] = DNCH_AUAU_MB

if DNCH_AUAU_MB_ERR is not None:
    df_direct_gamma["dNch_err"] = DNCH_AUAU_MB_ERR

optional_columns = [
    column
    for column in ["dNch", "dNch_err"]
    if column in df_direct_gamma.columns
]

df_direct_gamma[
    direct_columns + optional_columns
].to_csv(
    DIRECT_OUTPUT,
    index=False,
)

print("Saved:", DIRECT_OUTPUT)

Saved: output/final_arrays/auau_mb_direct_gamma.csv
